# Урок 6. GridSearchCV: подбор настроек модели

В уроке 5 ты подбирал глубину дерева **вручную**: цикл, таблица, график.
`GridSearchCV` делает то же самое сам и сразу для нескольких настроек.

Слова:
- **гиперпараметр** — настройка модели, которую мы задаём до обучения (`max_depth`, `min_samples_leaf`);
- **сетка** (grid) — список значений, которые надо попробовать;
- **GridSearchCV** — перебирает все комбинации и для каждой считает кросс-валидацию.

## 0. Подготовка

То же, что в уроке 5. Новое: сразу откладываем **тест** (179 человек). Зачем — поймёшь в части 3.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV

df = pd.read_csv("../data/titanic.csv")
y = df["Survived"]

df["Title"] = (df["Name"].str.split(", ").str[1].str.split(" ").str[0].str.rstrip("."))
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

num_cols = ["Pclass", "Age", "Fare", "FamilySize"]
cat_cols = ["Sex", "Embarked", "Title"]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat_cols),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 80% на учёбу и подбор, 20% прячем до самого конца
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)
print(len(X_train), len(X_test))

712 179


## 1. Одна настройка: глубина дерева

Сначала повторим урок 5, но без цикла.

Две новые вещи:
1. **Имена шагов.** В `Pipeline` мы сами называем шаги: `"prep"` и `"model"`.
2. **Имя настройки = шаг + два подчёркивания + настройка.** Глубина дерева в шаге `model` называется `model__max_depth`.

Сколько обучений? 8 значений глубины × 5 частей кросс-валидации = **40**.

In [2]:
pipe = Pipeline([("prep", preprocess),
                 ("model", DecisionTreeClassifier(random_state=42))])

param_grid = {"model__max_depth": [1, 2, 3, 4, 5, 6, 8, 10]}

search = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy")
search.fit(df, y)

print("лучшая настройка:", search.best_params_)
print(f"cv accuracy:      {search.best_score_:.3f}")

лучшая настройка: {'model__max_depth': 4}
cv accuracy:      0.827


**Вывод:** `GridSearchCV` выбрал глубину 4, cv = 0.827. Это **тот же ответ**, что на твоём графике из задания 2 урока 5.
Только код короче: цикл и график не нужны.

## 2. Что внутри: таблица `cv_results_`

Все результаты лежат в `search.cv_results_`. Превращаем в таблицу.

In [3]:
results = pd.DataFrame(search.cv_results_)
results[["param_model__max_depth", "mean_test_score", "std_test_score", "rank_test_score"]].round(3)

,param_model__max_depth,mean_test_score,std_test_score,rank_test_score
0,1,0.777,0.014,8
1,2,0.778,0.025,7
2,3,0.826,0.013,2
3,4,0.827,0.017,1
4,5,0.826,0.021,3
5,6,0.820,0.022,5
6,8,0.823,0.030,4
7,10,0.819,0.021,6


**Как читать:**
- `mean_test_score` — среднее по 5 частям (это твой cv).
- `std_test_score` — разброс по частям.
- `rank_test_score` — место. 1 = лучший.

**Вывод:** глубины 3, 4 и 5 дают 0.826, 0.827 и 0.826. Разница 0.001 намного меньше разброса (0.013–0.021). Значит, «лучшая глубина 4» — это не точный ответ, а «где-то 3–5». Ты это уже увидел на графике.

## 3. Две настройки сразу: случайный лес

Теперь сетка из двух настроек. Перебираются **все комбинации**:
4 значения глубины × 4 значения `min_samples_leaf` = 16 комбинаций × 5 частей = **80 обучений** (в каждом 300 деревьев).

- `n_jobs=-1` — использовать все ядра процессора (будет быстрее).
- Ищем только на `X_train` (712 человек). Тест не трогаем.

In [4]:
forest = Pipeline([("prep", preprocess),
                   ("model", RandomForestClassifier(n_estimators=300, random_state=42))])

param_grid = {
    "model__max_depth":        [4, 6, 8, None],
    "model__min_samples_leaf": [1, 2, 5, 10],
}

search = GridSearchCV(forest, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
search.fit(X_train, y_train)

print("лучшая настройка:", search.best_params_)
print(f"cv accuracy:      {search.best_score_:.3f}")

лучшая настройка: {'model__max_depth': 4, 'model__min_samples_leaf': 1}
cv accuracy:      0.830


In [5]:
results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
results[["param_model__max_depth", "param_model__min_samples_leaf",
         "mean_test_score", "std_test_score", "rank_test_score"]].head(6).round(3)

,param_model__max_depth,param_model__min_samples_leaf,mean_test_score,std_test_score,rank_test_score
0,4,1,0.830,0.028,1
1,4,2,0.829,0.030,2
2,4,5,0.829,0.030,2
5,6,2,0.824,0.020,4
14,None,5,0.824,0.024,5
10,8,5,0.824,0.024,5


**Вывод:** лучшая комбинация: глубина 4, `min_samples_leaf=1`, cv = 0.830.
Но смотри на первые три строки: 0.830, 0.829, 0.829. Разница 0.001, а разброс 0.03. Это **шум**.
Практический вывод: из почти одинаковых комбинаций выбирай ту, что **проще** (здесь — `min_samples_leaf=5` вместо 1: дерево менее склонно зубрить).

## 4. Ловушка: `best_score_` завышен

Мы попробовали 16 комбинаций и взяли лучшую. Лучшая комбинация часто лучшая **отчасти из-за везения** на этих 5 частях.
Поэтому `best_score_` — оптимистичная оценка.

Честная проверка: один раз посмотреть на **тест**, который поиск не видел.

После `fit` объект `search` уже содержит лучшую модель, переобученную на всём `X_train`. Её можно вызывать как обычную модель.

In [6]:
# обычный лес без подбора (настройки по умолчанию)
plain = Pipeline([("prep", preprocess),
                  ("model", RandomForestClassifier(n_estimators=300, random_state=42))])
plain.fit(X_train, y_train)

print(f"cv после подбора (best_score_): {search.best_score_:.3f}")
print(f"тест, лес после подбора:        {search.score(X_test, y_test):.3f}")
print(f"тест, лес без подбора:          {plain.score(X_test, y_test):.3f}")

n = len(X_test)
print("угадано людей (подбор / без подбора):",
      round(search.score(X_test, y_test) * n), "/", round(plain.score(X_test, y_test) * n), "из", n)

cv после подбора (best_score_): 0.830
тест, лес после подбора:        0.816
тест, лес без подбора:          0.810
угадано людей (подбор / без подбора): 146 / 145 из 179


**Вывод:**
- cv после подбора 0.830, а на тесте 0.816. Тест ниже, как и ожидали от оптимистичного `best_score_`. Но тест из 179 человек сам шумный (±0.03), поэтому по одному сравнению точно сказать нельзя. Важно запомнить: **итоговое число = тест, а не `best_score_`**.
- Подбор против обычного леса: 0.816 против 0.810. Это разница в **1 человека** из 179.
- Значит, на этих данных подбор почти ничего не дал. Модель уже была близка к потолку. Для Титаника это нормально: данных мало (891 строка).

**Правило:** подбор настроек почти никогда не творит чудес. Большие улучшения дают **признаки** (урок 4: титул и семья дали +4 пункта). Настройки дают доли процента.

## 5. Шпаргалка

```python
search = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
search.fit(X_train, y_train)        # перебор + переобучение лучшей на всём X_train

search.best_params_                 # лучшая комбинация
search.best_score_                  # cv лучшей (оптимистично)
search.cv_results_                  # все результаты (в DataFrame)
search.score(X_test, y_test)        # честная проверка на тесте
search.predict(X_new)               # предсказание лучшей моделью
```

Число обучений = (число комбинаций в сетке) × (число частей cv).
Сетка 3 × 4 × 2 с `cv=5` = 24 комбинации × 5 = 120 обучений.

---
# 🏠 Домашка

## Задание 1. Сколько обучений?

Без кода посчитай (формула в шпаргалке):
- а) сетка `max_depth` = [2, 4, 6, 8], `min_samples_leaf` = [1, 5, 10], `cv=5`;
- б) та же сетка, но `cv=10`.

## Задание 2. Логистическая регрессия: настройка `C`

`C` — сила «штрафа» за большие баллы: маленький `C` = модель проще. Переберите `C` = [0.01, 0.1, 1, 10, 100].
Логистической регрессии нужно масштабирование, поэтому возьми этот `preprocess_scaled`:

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

preprocess_scaled = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat_cols),
])
```

Что сделать:
1. Собери `Pipeline` с шагами `"prep"` и `"model"` (`LogisticRegression(max_iter=2000)`).
2. Имя настройки: подумай сам (шаг + `__` + `C`).
3. Запусти `GridSearchCV` на `X_train`, `y_train`.
4. Выведи `best_params_`, `best_score_` и результат на тесте.
5. **Вывод словами:** какой `C` лучший? Отличается ли он заметно от соседних? Сравни тест с лесом из части 4 (0.816).

## Задание 3. Подбор нестабилен?

Повтори часть 1 (дерево, глубина) на **тренировочной части**, но с разными `random_state` в `train_test_split`: 0, 1, 2, 3, 4.
Для каждого выведи `best_params_` и результат на тесте.
**Вывод словами:** всегда ли лучшая глубина одна и та же? Что это говорит про «лучшая глубина = 4»?

## Задание 4. Вопрос без кода

Друг сделал `GridSearchCV` и написал в резюме: «моя модель угадывает 0.830». Что ты ему скажешь про это число и что предложишь сделать?

**Ответ:**

### Ответ на задание 1

Число обучений = (значения 1-й настройки) × (значения 2-й настройки) × (число частей cv).

- а) 4 × 3 × 5 = **60** обучений;
- б) 4 × 3 × 10 = **120** обучений.

In [7]:
# задание 2: логистическая регрессия, настройка C
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

preprocess_scaled = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat_cols),
])

param_grid_lr = {"model__C": [0.01, 0.1, 1, 10, 100]}

pipe_lr = Pipeline([
    ("prep", preprocess_scaled),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

search_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=cv, scoring="accuracy", n_jobs=-1)
search_lr.fit(X_train, y_train)

test_acc = search_lr.score(X_test, y_test)
print("лучшая настройка:", search_lr.best_params_)
print(f"cv (best_score_): {search_lr.best_score_:.3f}")
print(f"тест:             {test_acc:.3f}  ({round(test_acc * len(X_test))} из {len(X_test)})")

# тест для C=1 и C=10 (у них одинаковый cv)
for C in [1, 10]:
    m = Pipeline([("prep", preprocess_scaled),
                  ("model", LogisticRegression(C=C, max_iter=1000, random_state=42))])
    m.fit(X_train, y_train)
    print(f"C={C}: тест {m.score(X_test, y_test):.3f}")

results_lr = pd.DataFrame(search_lr.cv_results_)
results_lr[["param_model__C", "mean_test_score", "std_test_score", "rank_test_score"]].round(3)

лучшая настройка: {'model__C': 10}
cv (best_score_): 0.820
тест:             0.838  (150 из 179)
C=1: тест 0.832
C=10: тест 0.838


,param_model__C,mean_test_score,std_test_score,rank_test_score
0,0.01,0.802,0.017,5
1,0.10,0.816,0.022,4
2,1.00,0.820,0.020,2
3,10.00,0.820,0.027,1
4,100.00,0.819,0.026,3


### Ответ на задание 2

Лучшее `C = 10` (cv 0.820, тест 0.838 = 150 из 179).

- `C = 1` и `C = 10` равны по cv (0.820). У `C = 1` разброс меньше (0.020 против 0.027), и модель проще (меньший `C` = проще). Я бы выбрал **C = 1** (тест 0.832, 149 из 179).
- `C = 0.01` хуже (0.802): модель слишком простая (недообучение).
- Обучений: 5 значений `C` × 5 частей = **25**.
- Сравнение с лесом из части 4: на тесте 0.838 против 0.816. Это разница в 4 человека из 179, а тест шумный (±0.03). Нельзя сказать, что регрессия лучше. Можно сказать: простая модель **не хуже** сложной.

In [8]:
# задание 3: подбор глубины на разных разбиениях
param_grid_depth = {"model__max_depth": [1, 2, 3, 4, 5, 6, 8, 10]}
seed_results = []

for split_seed in [0, 1, 2, 3, 4]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        df, y, test_size=0.2, random_state=split_seed, stratify=y
    )

    pipe_tree = Pipeline([
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(random_state=42)),
    ])

    search_tree = GridSearchCV(pipe_tree, param_grid_depth, cv=cv, scoring="accuracy")
    search_tree.fit(X_tr, y_tr)

    seed_results.append({
        "split_random_state": split_seed,
        "best_depth": search_tree.best_params_["model__max_depth"],
        "test_accuracy": search_tree.score(X_te, y_te),
    })

pd.DataFrame(seed_results).round(3)

,split_random_state,best_depth,test_accuracy
0,0,4,0.821
1,1,4,0.827
2,2,3,0.838
3,3,3,0.844
4,4,3,0.827


### Ответ на задание 3

Лучшая глубина **не всегда 4**: при `random_state` 0 и 1 лучшая 4, при 2, 3 и 4 лучшая 3.

Почему так:
- `random_state` — это просто **номер разбиения**, а не шкала. Другой номер = другие 179 человек в тесте и другие 712 в учёбе. Поэтому и `test_accuracy` разная (0.821–0.844).
- В части 1 у глубин 3, 4 и 5 cv почти одинаковый (0.826, 0.827, 0.826). Когда разница меньше шума, на каждом новом разбиении побеждает другая глубина.

**Вывод:** «лучшая глубина = 4» неточно. Правильнее: лучшая глубина **3–4**.

### Ответ на задание 4

**Что сказать другу:** «Ты взял лучшую из 16 комбинаций на одних и тех же частях. Часть этого результата — везение, число 0.830 завышено. Это `best_score_`, оно оптимистичное. Проверь на тесте, который поиск не видел, и **это** число пиши в резюме». У нас на тесте вышло 0.816.

**Аналогия с монетами.** Даже если все 16 монет честные, за 5 бросков у одной случайно выпадет больше орлов. Если выбрать именно её, на новых бросках её преимущество скорее всего исчезнет: ей просто повезло в первой серии. `best_score_` — это «лучшая монета», а тест — «новые броски».